# Cloud Kitchen P&L — Data Analysis Notebook
**Author:** Tathagata Ghosh  
**Python Version:** 3.14  
**Packages:** streamlit==1.45.1 | pandas==2.2.3 | plotly==5.24.1 | openpyxl==3.1.5  
**Live Dashboard:** https://kitchen-dashboard.streamlit.app  
**GitHub:** https://github.com/tathagat17/kitchen-dashboard

---
This notebook covers data preparation, exploration and key insights for the Cloud Kitchen P&L dashboard.  
Data: **344 stores | 5 cities | 4 zones | 6 months (Oct 2023 – Mar 2024)**

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_excel('Untitled_spreadsheet.xlsx', header=1)
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())

## 2. Data Preparation (Matching app.py logic exactly)

In [ ]:
# Computed columns — same as app.py
df['GM%']       = (df['GROSS MARGIN']   / df['NET REVENUE'] * 100).round(2)
df['CM%']       = (df['KITCHEN EBITDA'] / df['NET REVENUE'] * 100).round(2)
df['EBITDA%']   = (df['KITCHEN EBITDA'] / df['NET REVENUE'] * 100).round(2)
df['VARIANCE%'] = (df['VARIANCE']        / df['NET REVENUE'] * 100).round(4)

# Variance buckets — same as app.py
def variance_bucket(v):
    if v < 2:   return '(a) Var < 2%'
    elif v < 3: return '(b) Var 2% to 3%'
    elif v < 5: return '(c) Var 3% to 5%'
    else:       return '(d) Var > 5%'
df['VARIANCE BUCKET'] = df['VARIANCE%'].apply(variance_bucket)

# Revenue bands — same as app.py
df['REVENUE BAND'] = pd.cut(
    df['NET REVENUE'] / 100000,
    bins=[0, 15, 25, 35, 45, float('inf')],
    labels=['(a) Below INR 15 lacs','(b) INR 15 to 25 lacs',
            '(c) INR 25 to 35 lacs','(d) INR 35 to 45 lacs','(e) Above INR 45 lacs']
)

# Month ordering — same as app.py
month_order = ['Oct-2023','Nov-2023','Dec-2023','Jan-2024','Feb-2024','Mar-2024']
df['MONTH'] = pd.Categorical(df['MONTH'], categories=month_order, ordered=True)
df = df.sort_values('MONTH')

print('Variance Bucket Distribution:')
print(df['VARIANCE BUCKET'].value_counts())
print('\nRevenue Band Distribution:')
print(df['REVENUE BAND'].value_counts().sort_index())

## 3. Dataset Overview

In [ ]:
print(f'Total Records     : {len(df)}')
print(f'Unique Stores     : {df["STORE"].nunique()}')
print(f'Cities            : {sorted(df["CITY"].unique().tolist())}')
print(f'Zones             : {df["ZONE MAPPING"].unique().tolist()}')
print(f'Avg Net Revenue   : ₹{df["NET REVENUE"].mean()/100000:.1f}L')
print(f'Avg EBITDA        : ₹{df["KITCHEN EBITDA"].mean()/100000:.1f}L')
print(f'Avg GM%           : {df["GM%"].mean():.1f}%')
print(f'Avg CM%           : {df["CM%"].mean():.1f}%')
print(f'EBITDA +ve        : {(df["EBITDA CATEGORY"]=="EBITDA +ve").mean()*100:.1f}%')
print(f'EBITDA -ve        : {(df["EBITDA CATEGORY"]=="EBITDA -ve").mean()*100:.1f}%')
df[['NET REVENUE','GROSS MARGIN','KITCHEN EBITDA','GM%','CM%','VARIANCE%']].describe().round(2)

## 4. Dashboard 1 — Kitchen Snapshot Pivot Table

In [ ]:
pivot_d1 = df.pivot_table(
    index=['STORE','CITY','ZONE MAPPING'],
    columns='MONTH',
    values=['NET REVENUE','GM%','CM%','KITCHEN EBITDA','EBITDA%'],
    aggfunc='mean'
).round(2)
pivot_d1.columns = [f'{c[0]} | {c[1]}' for c in pivot_d1.columns]
pivot_d1 = pivot_d1.reset_index()
print('Pivot shape:', pivot_d1.shape)
pivot_d1.head(3)

## 5. City-Level Performance

In [ ]:
city_perf = df.groupby('CITY').agg(
    Stores        =('STORE','nunique'),
    Avg_Rev_L     =('NET REVENUE', lambda x: round(x.mean()/100000,1)),
    Avg_EBITDA_L  =('KITCHEN EBITDA', lambda x: round(x.mean()/100000,1)),
    Avg_GM_pct    =('GM%','mean'),
    Avg_CM_pct    =('CM%','mean'),
    Avg_Var_pct   =('VARIANCE%','mean')
).round(2).reset_index()
print(city_perf.to_string(index=False))

fig = px.bar(city_perf, x='CITY', y='Avg_EBITDA_L',
    color='Avg_EBITDA_L', color_continuous_scale='RdYlGn',
    title='Average EBITDA by City (₹ Lakhs)', text='Avg_EBITDA_L')
fig.update_layout(plot_bgcolor='white')
fig.show()

**Insight:** Ahmedabad leads EBITDA (₹7.2L). Mumbai lowest (₹6.3L) despite similar revenue — higher costs suspected.

## 6. Monthly Trend

In [ ]:
monthly = df.groupby('MONTH', observed=True).agg(
    Avg_Revenue=('NET REVENUE','mean'),
    Avg_EBITDA =('KITCHEN EBITDA','mean'),
    Avg_GM_pct =('GM%','mean'),
    Avg_CM_pct =('CM%','mean')
).round(2).reset_index()
print(monthly.to_string(index=False))

fig = make_subplots(rows=1, cols=2, subplot_titles=['Net Revenue Trend','EBITDA Trend'])
fig.add_trace(go.Scatter(x=monthly['MONTH'], y=monthly['Avg_Revenue'],
    mode='lines+markers', name='Revenue', line=dict(color='#3498db',width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=monthly['MONTH'], y=monthly['Avg_EBITDA'],
    mode='lines+markers', name='EBITDA', line=dict(color='#2ecc71',width=3)), row=1, col=2)
fig.update_layout(title='Monthly Revenue & EBITDA Trend', plot_bgcolor='white', height=400)
fig.show()

**Insight:** January 2024 peak month. November 2023 weakest — post-festive dip.

## 7. Profitable vs Loss-Making Stores

In [ ]:
print(df['EBITDA CATEGORY'].value_counts())
print(f'\n52.1% of store-months are LOSS MAKING — critical concern')

pie_df = df['EBITDA CATEGORY'].value_counts().reset_index()
pie_df.columns = ['Category','Count']
fig = px.pie(pie_df, names='Category', values='Count', hole=0.4,
    color='Category',
    color_discrete_map={'EBITDA +ve':'#2ecc71','EBITDA -ve':'#e74c3c'},
    title='Profitable vs Loss-Making Stores')
fig.show()

## 8. Zone Performance

In [ ]:
zone_perf = df.groupby('ZONE MAPPING').agg(
    Avg_Rev_L    =('NET REVENUE', lambda x: round(x.mean()/100000,1)),
    Avg_EBITDA_L =('KITCHEN EBITDA', lambda x: round(x.mean()/100000,1)),
    Avg_CM_pct   =('CM%','mean')
).round(2).reset_index()
print(zone_perf.to_string(index=False))

fig = px.bar(zone_perf, x='ZONE MAPPING', y='Avg_CM_pct',
    color='Avg_CM_pct', color_continuous_scale='Blues',
    title='Average CM% by Zone', text='Avg_CM_pct')
fig.update_layout(plot_bgcolor='white')
fig.show()

**Insight:** East best (17.2% CM%), North worst (16.0%). 1.2% gap significant at scale.

## 9. Top & Bottom Stores

In [ ]:
store_perf = df.groupby('STORE').agg(
    Avg_Rev_L   =('NET REVENUE', lambda x: round(x.mean()/100000,1)),
    Avg_EBITDA_L=('KITCHEN EBITDA', lambda x: round(x.mean()/100000,1)),
    Avg_CM_pct  =('CM%','mean')
).round(2)
print('TOP 5 BY REVENUE:')
print(store_perf.sort_values('Avg_Rev_L', ascending=False).head(5))
print('\nBOTTOM 5 BY EBITDA:')
print(store_perf.sort_values('Avg_EBITDA_L').head(5))

top10 = store_perf.sort_values('Avg_EBITDA_L', ascending=False).head(10).reset_index()
fig = px.bar(top10, x='STORE', y='Avg_EBITDA_L',
    color='Avg_EBITDA_L', color_continuous_scale='Greens',
    title='Top 10 Stores by EBITDA', text='Avg_EBITDA_L')
fig.update_layout(plot_bgcolor='white', xaxis_tickangle=-30)
fig.show()

## 10. Dashboard 2 — Variance PNL Pivots

In [ ]:
# Sub-Dashboard 2a — Avg Variance % by Revenue Cohort
rev_cohort_order = ['INR 20 to 30 lacs','INR 30 to 40 lacs','More than 40 lacs']
pivot2a = df.pivot_table(index='REVENUE COHORT', columns='MONTH',
    values='VARIANCE%', aggfunc='mean').round(4)
pivot2a = pivot2a.reindex([r for r in rev_cohort_order if r in pivot2a.index])
grand_row = pd.DataFrame(df.groupby('MONTH', observed=True)['VARIANCE%'].mean().round(4)).T
grand_row.index = ['Grand Total']
pivot2a = pd.concat([pivot2a, grand_row])
display = pivot2a.apply(lambda col: col.map(lambda x: f'{x:.2f}%' if pd.notnull(x) else '-'))
print('Sub-Dashboard 1 — Avg Variance % by Revenue Cohort:')
print(display.to_string())

In [ ]:
# Sub-Dashboard 2b — Store Count by Revenue Band
rev_band_order = ['(a) Below INR 15 lacs','(b) INR 15 to 25 lacs',
    '(c) INR 25 to 35 lacs','(d) INR 35 to 45 lacs','(e) Above INR 45 lacs']
pivot2b = df.pivot_table(index='REVENUE BAND', columns='MONTH',
    values='STORE', aggfunc='count').fillna(0).astype(int)
pivot2b = pivot2b.reindex([r for r in rev_band_order if r in pivot2b.index])
grand_b = pd.DataFrame(pivot2b.sum()).T
grand_b.index = ['Grand Total']
pivot2b = pd.concat([pivot2b, grand_b])
print('Sub-Dashboard 2 — Store Count by Revenue Band:')
print(pivot2b.to_string())

## 11. Variance (Food Wastage) Heatmap

In [ ]:
print('Variance % by City:')
print(df.groupby('CITY')['VARIANCE%'].mean().round(4))

heat_df = df.pivot_table(index='CITY', columns='MONTH',
    values='VARIANCE%', aggfunc='mean').round(4)
fig = px.imshow(heat_df, color_continuous_scale='RdYlGn_r',
    text_auto='.2f', title='Food Wastage Heatmap — City × Month')
fig.show()

**Insight:** Pune highest wastage (0.63%), Hyderabad most efficient (0.61%). Pune Jan-2024 = 0.69% peak.

## 12. Performance Optimization — Caching

In `app.py`, data loading is cached using:
```python
@st.cache_data(ttl=300)  # 5 minute cache
def load_data():
    df = pd.read_excel('Untitled_spreadsheet.xlsx', header=1)
    return df
```
- Prevents re-reading Excel on every user interaction
- TTL=300 means auto-refresh every 5 minutes for real-time pipelines
- Manual refresh via sidebar button clears cache instantly

## 13. Key Insights Summary

| # | Insight | Severity |
|---|---------|----------|
| 1 | 52.1% stores EBITDA negative — majority loss-making | 🔴 Critical |
| 2 | Ahmedabad best city — highest EBITDA ₹7.2L, lowest wastage | 🟢 Positive |
| 3 | Mumbai underperforms — lowest EBITDA ₹6.3L vs similar revenue | 🔴 Action needed |
| 4 | January 2024 peak — highest revenue ₹35.6L and EBITDA ₹7.3L | 🟢 Seasonal |
| 5 | North zone lowest CM% 16.0% vs East zone 17.2% | 🟡 Investigate |
| 6 | Pune highest wastage 0.63% — Jan-2024 peak 0.69% | 🟡 Fix ops |
| 7 | Andrews Reed & Silva chronic loss — avg EBITDA -₹1.3L | 🔴 Immediate review |
| 8 | All variance <1% but recovery possible at scale | 🟡 Opportunity |

### Recommendations
1. Investigate Mumbai for cost inefficiencies
2. Replicate Ahmedabad best practices
3. North zone cost optimization program
4. Pune food wastage reduction — target below 0.55%
5. Monthly review for all EBITDA -ve stores
6. Leverage January demand pattern across all cities